# Data Cleaning Notebook

This notebook documents how the cleaned import files for `film_db` are produced from the given coursework dataset.

**Design principle:** the assignment dataset is fixed. The code does not invent, redesign, or enrich film/person/review/role facts. It only performs traceable cleaning needed for PostgreSQL import, type consistency, and referential integrity.

The notebook outputs:

- `films_cleaned.csv`
- `people_cleaned.csv`
- `reviews_cleaned.csv`
- `roles_cleaned.csv`
- `data_quality_log.csv`
- `cleaning_summary.csv`

Excel row numbers in the data-quality log use the original worksheet numbering, where row 1 is the header, so a pandas index of `0` corresponds to Excel row `2`.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)

# Raw files provided for Assessment 002.
REQUIRED_FILES = {
    'films': 'films.xlsx',
    'people': 'people.xlsx',
    'reviews': 'reviews.xlsx',
    'roles': 'roles.xlsx',
}

def find_raw_dir():
    """Find the folder that contains all four raw coursework files."""
    candidates = [Path.cwd(), Path('/mnt/data')]
    for directory in candidates:
        if all((directory / filename).exists() for filename in REQUIRED_FILES.values()):
            return directory
    raise FileNotFoundError(
        'Could not find all four raw xlsx files. Put them in the same folder as this notebook, '
        'or edit RAW_DIR manually.'
    )

RAW_DIR = find_raw_dir()
OUT_DIR = RAW_DIR / 'cleaned_output'
OUT_DIR.mkdir(exist_ok=True)

print(f'Raw data folder: {RAW_DIR}')
print(f'Output folder: {OUT_DIR}')

Raw data folder: c:\Users\xxc\Desktop\Data Cleaning
Output folder: c:\Users\xxc\Desktop\Data Cleaning\cleaned_output


## 1. Load the given dataset

The four source files are loaded exactly as provided. The original dataframes are kept unchanged so that all cleaning decisions can be compared against the raw data.

In [3]:
films_raw = pd.read_excel(RAW_DIR / REQUIRED_FILES['films'])
people_raw = pd.read_excel(RAW_DIR / REQUIRED_FILES['people'])
reviews_raw = pd.read_excel(RAW_DIR / REQUIRED_FILES['reviews'])
roles_raw = pd.read_excel(RAW_DIR / REQUIRED_FILES['roles'])

raw_summary = pd.DataFrame({
    'file': ['films', 'people', 'reviews', 'roles'],
    'rows': [len(films_raw), len(people_raw), len(reviews_raw), len(roles_raw)],
    'columns': [films_raw.shape[1], people_raw.shape[1], reviews_raw.shape[1], roles_raw.shape[1]],
})
raw_summary

,file,rows,columns
0,films,4949,9
1,people,8397,4
2,reviews,4968,6
3,roles,19791,4


In [4]:
print('films columns:', list(films_raw.columns))
print('people columns:', list(people_raw.columns))
print('reviews columns:', list(reviews_raw.columns))
print('roles columns:', list(roles_raw.columns))

films columns: ['id', 'title', 'release_year', 'place_name', 'duration', 'langauge', 'certification', 'gross', 'budget']
people columns: ['id', 'name', 'birthdate', 'deathdate']
reviews columns: ['id', 'film_id', 'num_critic', 'imdb_score', 'num_votes', 'facebook_likes']
roles columns: ['id', 'film_id', 'people_id', 'role']


## 2. Helper functions

These functions standardise text, integer-like numeric columns, and dates. The date function is important because the `people` file contains both Excel serial numbers and text-format dates.

In [5]:
def clean_text_series(series: pd.Series) -> pd.Series:
    """Trim strings and treat empty strings as missing values."""
    return series.astype('string').str.strip().replace({'': pd.NA})


def to_nullable_int(series: pd.Series) -> pd.Series:
    """Convert numeric-like values to pandas nullable integers, preserving missing values."""
    return pd.to_numeric(series, errors='coerce').round().astype('Int64')


def excel_or_text_date_to_iso(value):
    """Convert Excel serial dates or text dates to ISO YYYY-MM-DD strings.

    Excel stores many dates as serial day counts. PostgreSQL DATE import is more reliable when all
    dates are exported as a single ISO format.
    """
    if pd.isna(value):
        return pd.NA
    if isinstance(value, (int, float, np.integer, np.floating)) and not pd.isna(value):
        converted = pd.to_datetime(float(value), unit='D', origin='1899-12-30', errors='coerce')
    else:
        converted = pd.to_datetime(str(value).strip(), errors='coerce')
    if pd.isna(converted):
        return pd.NA
    return converted.strftime('%Y-%m-%d')


def excel_rows(index_like):
    """Convert pandas zero-based row indexes to Excel worksheet row numbers."""
    return [int(i) + 2 for i in index_like]

## 3. Clean `films`

Cleaning actions:

- Rename the misspelled column `langauge` to `language`.
- Trim text fields.
- Convert integer-like numeric fields to integer types while preserving blanks.
- Keep all film rows because no film record needs to be fabricated or deleted.

In [6]:
films = films_raw.copy()

# Schema-level correction: spelling correction only, not a data-value change.
films = films.rename(columns={'langauge': 'language'})

for col in ['title', 'place_name', 'language', 'certification']:
    films[col] = clean_text_series(films[col])

for col in ['id', 'release_year', 'duration', 'gross', 'budget']:
    films[col] = to_nullable_int(films[col])

films = films[['id', 'title', 'release_year', 'place_name', 'duration', 'language', 'certification', 'gross', 'budget']]
films.head()

,id,title,release_year,place_name,duration,language,certification,gross,budget
0,1,Intolerance: Love's Struggle Throughout the Ages,1916,USA,123,<NA>,Not Rated,<NA>,385907
1,2,Over the Hill to the Poorhouse,1920,USA,110,<NA>,<NA>,3000000,100000
2,3,The Big Parade,1925,USA,151,<NA>,Not Rated,<NA>,245000
3,4,Metropolis,1927,Germany,145,German,Not Rated,26435,6000000
4,5,Pandora's Box,1929,Germany,110,German,Not Rated,9950,<NA>


## 4. Clean `people`

Cleaning actions:

- Trim names.
- Correct the corrupted UTF-8 replacement character in `Jordi Moll�` to `Jordi Mollà`.
- Standardise `birthdate` and `deathdate` to `YYYY-MM-DD` for PostgreSQL `DATE` import.
- Keep all people rows because the issue is formatting/encoding, not invalid identity data.

In [7]:
people = people_raw.copy()

people['name'] = clean_text_series(people['name'])
people['name'] = people['name'].str.replace('Jordi Moll�', 'Jordi Mollà', regex=False).str.strip()

people['birthdate'] = people['birthdate'].apply(excel_or_text_date_to_iso).astype('string')
people['deathdate'] = people['deathdate'].apply(excel_or_text_date_to_iso).astype('string')
people['id'] = to_nullable_int(people['id'])

people = people[['id', 'name', 'birthdate', 'deathdate']]
people.head()

,id,name,birthdate,deathdate
0,1,50 Cent,1975-07-06,<NA>
1,2,A. Michael Baldwin,1963-04-04,<NA>
2,3,A. Raven Cruz,<NA>,<NA>
3,4,A.J. Buckley,1978-02-09,<NA>
4,5,A.J. DeLucia,<NA>,<NA>


In [8]:
# Evidence for the mixed-date issue reported in Q3.
birth_text_mask = people_raw['birthdate'].notna() & pd.to_numeric(people_raw['birthdate'], errors='coerce').isna()
death_text_mask = people_raw['deathdate'].notna() & pd.to_numeric(people_raw['deathdate'], errors='coerce').isna()
date_issue_mask = birth_text_mask | death_text_mask

print('Rows with text-form dates:', int(date_issue_mask.sum()))
print('Example Excel rows:', excel_rows(people_raw.index[date_issue_mask])[:10])

Rows with text-form dates: 83
Example Excel rows: [62, 247, 278, 667, 680, 754, 805, 818, 1075, 1115]


## 5. Clean `reviews`

Cleaning actions:

- Convert ID/count fields to integer-compatible columns.
- Round `imdb_score` to one decimal, matching the intended rating scale and the PostgreSQL `NUMERIC(3,1)` design.
- Delete review rows with `NULL film_id`, because a review must belong to exactly one film.
- Delete orphan review rows whose `film_id` does not exist in `films.id`, because missing films cannot be fabricated.

In [9]:
reviews = reviews_raw.copy()

for col in ['id', 'film_id', 'num_critic', 'num_votes', 'facebook_likes']:
    reviews[col] = to_nullable_int(reviews[col])

reviews['imdb_score'] = pd.to_numeric(reviews['imdb_score'], errors='coerce').round(1)

film_ids = set(films['id'].dropna().astype(int))
reviews_null_fk_mask = reviews['film_id'].isna()
reviews_orphan_fk_mask = reviews['film_id'].notna() & ~reviews['film_id'].isin(film_ids)

reviews_removed = reviews[reviews_null_fk_mask | reviews_orphan_fk_mask].copy()
reviews = reviews[~(reviews_null_fk_mask | reviews_orphan_fk_mask)].copy()
reviews = reviews[['id', 'film_id', 'num_critic', 'imdb_score', 'num_votes', 'facebook_likes']]

print('Reviews removed because film_id is NULL:', int(reviews_null_fk_mask.sum()))
print('Reviews removed because film_id is absent from films.id:', int(reviews_orphan_fk_mask.sum()))
print('Removed review Excel rows:', excel_rows(reviews_removed.index))
reviews_removed.head()

Reviews removed because film_id is NULL: 19
Reviews removed because film_id is absent from films.id: 4
Removed review Excel rows: [56, 190, 552, 606, 1191, 1330, 1744, 1812, 2347, 2416, 2752, 2856, 3090, 3238, 3337, 3393, 3710, 3951, 4120, 4657, 4806, 4828, 4853]


,id,film_id,num_critic,imdb_score,num_votes,facebook_likes
54,4362,<NA>,<NA>,4.3,28,17
188,2926,<NA>,<NA>,5.2,118,30
550,4953,<NA>,<NA>,7.2,18,0
604,4490,<NA>,<NA>,7.0,8,65
1189,3473,<NA>,<NA>,4.3,30,16


## 6. Clean `roles`

Cleaning actions:

- Convert key fields to integer-compatible columns.
- Standardise the `role` value to lowercase text.
- Delete orphan role rows whose `film_id` is absent from `films.id`.
- Check whether any `people_id` is absent from `people.id`. In this dataset, no invalid `people_id` rows are found.

In [10]:
roles = roles_raw.copy()

for col in ['id', 'film_id', 'people_id']:
    roles[col] = to_nullable_int(roles[col])

roles['role'] = clean_text_series(roles['role']).str.lower()
people_ids = set(people['id'].dropna().astype(int))

roles_orphan_film_mask = roles['film_id'].isna() | ~roles['film_id'].isin(film_ids)
roles_orphan_people_mask = roles['people_id'].isna() | ~roles['people_id'].isin(people_ids)

roles_removed = roles[roles_orphan_film_mask | roles_orphan_people_mask].copy()
roles = roles[~(roles_orphan_film_mask | roles_orphan_people_mask)].copy()
roles = roles[['id', 'film_id', 'people_id', 'role']]

print('Roles removed because film_id is absent from films.id:', int(roles_orphan_film_mask.sum()))
print('Roles removed because people_id is absent from people.id:', int(roles_orphan_people_mask.sum()))
print('Number of absent film ids referenced by roles:', roles_removed['film_id'].nunique())
roles_removed.head()

Roles removed because film_id is absent from films.id: 76
Roles removed because people_id is absent from people.id: 0
Number of absent film ids referenced by roles: 19


,id,film_id,people_id,role
2938,2939,736,424,actor
2939,2940,736,461,actor
2940,2941,736,1300,director
2941,2942,736,3076,actor
3154,3155,790,461,actor


## 7. Build the data-quality log

This table is the executable version of the Q3 cleaning evidence. It records the file, original Excel row number, problem, solution, and justification. It can be exported directly and used to support the report table.

In [11]:
issue_rows = []

# films issue: header/schema label
issue_rows.append([
    'films.xlsx',
    'Header row 1',
    'Column name misspelled as langauge.',
    'Renamed to language in the cleaned CSV and DDL.',
    'Improves schema clarity without changing data values.'
])

# people issue: corrupted character
issue_rows.append([
    'people.xlsx',
    '4110',
    'Corrupted replacement character and trailing space in Jordi Moll� .',
    'Corrected to Jordi Mollà and trimmed whitespace.',
    'Fixes an unexpected UTF-8 issue.'
])

# people issue: mixed date formats
date_issue_rows = excel_rows(people_raw.index[date_issue_mask])
date_example = date_issue_rows[0] if date_issue_rows else 'N/A'
issue_rows.append([
    'people.xlsx',
    f'{len(date_issue_rows)} rows, e.g., {date_example}',
    'Mixed date formats: some dates are text while most are Excel serial numbers.',
    'Converted all birthdate/deathdate values to YYYY-MM-DD.',
    'Required for PostgreSQL DATE import.'
])

# reviews issue: NULL film_id
null_review_rows = excel_rows(reviews_raw.index[reviews_raw['film_id'].isna()])
issue_rows.append([
    'reviews.xlsx',
    f'{len(null_review_rows)} rows, e.g., {null_review_rows[0] if null_review_rows else "N/A"}',
    'film_id is NULL.',
    'Deleted affected review rows.',
    'film_id is a mandatory FK and cannot be inferred safely.'
])

# reviews issue: orphan film_id
review_orphans_raw = reviews_raw[reviews_raw['film_id'].notna()].copy()
review_orphans_raw = review_orphans_raw[~review_orphans_raw['film_id'].astype(int).isin(film_ids)]
for film_id, group in review_orphans_raw.groupby('film_id', sort=True):
    rows = excel_rows(group.index)
    row_label = ', '.join(map(str, rows)) if len(rows) <= 4 else f'{rows[0]}-{rows[-1]}'
    issue_rows.append([
        'reviews.xlsx',
        row_label,
        f'film_id {int(film_id)} does not exist in films.id.',
        'Deleted the orphan review row(s).',
        'A missing film record cannot be fabricated.'
    ])

# reviews issue: rating precision standardisation
issue_rows.append([
    'reviews.xlsx',
    'All review rows with non-null imdb_score',
    'imdb_score is stored as a floating value and may display binary precision artifacts in PostgreSQL/Python.',
    'Rounded to one decimal and stored as NUMERIC(3,1).',
    'Preserves the original rating scale.'
])

# roles issue: orphan films, with two detailed examples plus a total row
role_orphans_raw = roles_raw[~roles_raw['film_id'].astype(int).isin(film_ids)].copy()
for film_id in sorted(role_orphans_raw['film_id'].unique())[:2]:
    rows = excel_rows(role_orphans_raw.index[role_orphans_raw['film_id'] == film_id])
    issue_rows.append([
        'roles.xlsx',
        f'{rows[0]}-{rows[-1]}',
        f'film_id {int(film_id)} does not exist in films.id.',
        'Deleted four orphan role rows.',
        'Prevents FK violation.'
    ])

issue_rows.append([
    'roles.xlsx',
    f'{len(role_orphans_raw)} rows total',
    f'Roles reference {role_orphans_raw["film_id"].nunique()} film ids absent from films.xlsx.',
    'Deleted all orphan role rows.',
    'No invalid people_id values were found.'
])

quality_log = pd.DataFrame(
    issue_rows,
    columns=['File Name', 'Excel Sheet Row No', 'Issue Identified', 'How Solved', 'Comments']
)
quality_log

,File Name,Excel Sheet Row No,Issue Identified,How Solved,Comments
0,films.xlsx,Header row 1,Column name misspelled as langauge.,Renamed to language in the cleaned CSV and DDL.,Improves schema clarity without changing data ...
1,people.xlsx,4110,Corrupted replacement character and trailing s...,Corrected to Jordi Mollà and trimmed whitespace.,Fixes an unexpected UTF-8 issue.
2,people.xlsx,"83 rows, e.g., 62",Mixed date formats: some dates are text while ...,Converted all birthdate/deathdate values to YY...,Required for PostgreSQL DATE import.
3,reviews.xlsx,"19 rows, e.g., 56",film_id is NULL.,Deleted affected review rows.,film_id is a mandatory FK and cannot be inferr...
4,reviews.xlsx,"2752, 4806",film_id 736 does not exist in films.id.,Deleted the orphan review row(s).,A missing film record cannot be fabricated.
5,reviews.xlsx,4120,film_id 1641 does not exist in films.id.,Deleted the orphan review row(s).,A missing film record cannot be fabricated.
6,reviews.xlsx,3090,film_id 5060 does not exist in films.id.,Deleted the orphan review row(s).,A missing film record cannot be fabricated.
7,reviews.xlsx,All review rows with non-null imdb_score,imdb_score is stored as a floating value and m...,Rounded to one decimal and stored as NUMERIC(3...,Preserves the original rating scale.
8,roles.xlsx,2940-2943,film_id 736 does not exist in films.id.,Deleted four orphan role rows.,Prevents FK violation.
9,roles.xlsx,3156-3159,film_id 790 does not exist in films.id.,Deleted four orphan role rows.,Prevents FK violation.


## 8. Validate cleaned data before export

The following assertions mirror the PostgreSQL constraints used in `ddl.sql`. If any assertion fails, the cleaned CSVs should not be imported until the problem is fixed.

In [12]:
# Primary-key checks
assert films['id'].notna().all() and films['id'].is_unique, 'films.id must be non-null and unique.'
assert people['id'].notna().all() and people['id'].is_unique, 'people.id must be non-null and unique.'
assert reviews['id'].notna().all() and reviews['id'].is_unique, 'reviews.id must be non-null and unique.'
assert roles['id'].notna().all() and roles['id'].is_unique, 'roles.id must be non-null and unique.'

# Foreign-key checks
assert reviews['film_id'].notna().all(), 'reviews.film_id must be non-null.'
assert reviews['film_id'].isin(film_ids).all(), 'Every reviews.film_id must exist in films.id.'
assert roles['film_id'].notna().all(), 'roles.film_id must be non-null.'
assert roles['film_id'].isin(film_ids).all(), 'Every roles.film_id must exist in films.id.'
assert roles['people_id'].notna().all(), 'roles.people_id must be non-null.'
assert roles['people_id'].isin(people_ids).all(), 'Every roles.people_id must exist in people.id.'

# Domain checks
assert reviews['imdb_score'].notna().all() and reviews['imdb_score'].between(0, 10).all(), 'imdb_score must be 0-10 and non-null.'
assert roles['role'].isin(['actor', 'director']).all(), "role must be either 'actor' or 'director'."

# Candidate-key style check used by the final DDL
assert roles.duplicated(['film_id', 'people_id', 'role']).sum() == 0, 'Duplicate film/person/role combinations found.'

print('All validation checks passed.')

All validation checks passed.


## 9. Export cleaned CSV files

The output files are encoded as UTF-8 and are ready for import into PostgreSQL through pgAdmin. Use the import order `films`, `people`, `reviews`, `roles` so that parent tables are imported before dependent tables.

In [13]:
outputs = {
    'films_cleaned.csv': films,
    'people_cleaned.csv': people,
    'reviews_cleaned.csv': reviews,
    'roles_cleaned.csv': roles,
    'data_quality_log.csv': quality_log,
}

for filename, dataframe in outputs.items():
    dataframe.to_csv(OUT_DIR / filename, index=False, encoding='utf-8', na_rep='')

cleaning_summary = pd.DataFrame({
    'table': ['films', 'people', 'reviews', 'roles'],
    'raw_rows': [len(films_raw), len(people_raw), len(reviews_raw), len(roles_raw)],
    'removed_rows': [len(films_raw) - len(films), len(people_raw) - len(people), len(reviews_raw) - len(reviews), len(roles_raw) - len(roles)],
    'cleaned_rows': [len(films), len(people), len(reviews), len(roles)],
    'main_reason_for_removal': [
        'No rows removed; header langauge renamed to language.',
        'No rows removed; name encoding and date formats standardised.',
        'Rows removed only when film_id was NULL or absent from films.id.',
        'Rows removed only when film_id was absent from films.id.'
    ]
})
cleaning_summary.to_csv(OUT_DIR / 'cleaning_summary.csv', index=False, encoding='utf-8')

print('Cleaned files exported to:', OUT_DIR)
cleaning_summary

Cleaned files exported to: c:\Users\xxc\Desktop\Data Cleaning\cleaned_output


,table,raw_rows,removed_rows,cleaned_rows,main_reason_for_removal
0,films,4949,0,4949,No rows removed; header langauge renamed to la...
1,people,8397,0,8397,No rows removed; name encoding and date format...
2,reviews,4968,23,4945,Rows removed only when film_id was NULL or abs...
3,roles,19791,76,19715,Rows removed only when film_id was absent from...


## 10. Final quick previews

These previews are for checking that the cleaned files have the expected columns and row counts before PostgreSQL import.

In [14]:
for name, df in [('films', films), ('people', people), ('reviews', reviews), ('roles', roles)]:
    print(f'\n{name}: {df.shape[0]} rows x {df.shape[1]} columns')
    print(df.head(3).to_string(index=False))


films: 4949 rows x 9 columns
 id                                            title  release_year place_name  duration language certification   gross  budget
  1 Intolerance: Love's Struggle Throughout the Ages          1916        USA       123     <NA>     Not Rated    <NA>  385907
  2                   Over the Hill to the Poorhouse          1920        USA       110     <NA>          <NA> 3000000  100000
  3                                   The Big Parade          1925        USA       151     <NA>     Not Rated    <NA>  245000

people: 8397 rows x 4 columns
 id               name  birthdate deathdate
  1            50 Cent 1975-07-06      <NA>
  2 A. Michael Baldwin 1963-04-04      <NA>
  3      A. Raven Cruz       <NA>      <NA>

reviews: 4945 rows x 6 columns
  id  film_id  num_critic  imdb_score  num_votes  facebook_likes
3934      588         432         7.1     203461           46000
3405      285         267         6.4     149998               0
 478       65          29   